# Circuit Motifs

Detect and characterize recurring circuit patterns: skip trigram circuits,
negative name movers, backup/redundancy circuits, signal boosting, and
automated motif cataloging.

References:
- Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"
- Wang et al. (2023) "Interpretability in the Wild"

## Why This Matters

Circuit motifs helps identify and characterize the specific subnetworks responsible for model behaviors. Finding circuits — the algorithms models actually implement — is the core goal of mechanistic interpretability.

**Key references:**
- [Wang et al. (2023) "Interpretability in the Wild: IOI"](https://arxiv.org/abs/2211.00593) — Detailed circuit analysis of indirect object identification
- [Conmy et al. (2023) "Towards Automated Circuit Discovery"](https://arxiv.org/abs/2304.14997) — Automated methods for circuit finding via edge pruning

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.circuit_motifs import (
    skip_trigram_detection,
    negative_mover_detection,
    backup_circuit_detection,
    signal_boosting_detection,
    motif_catalog,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([0, 5, 10, 15, 20, 25, 30])
metric_fn = lambda logits: float(logits[-1, 0] - logits[-1, 1])
print("Model ready")

## Skip Trigram Detection

In [ ]:
result = skip_trigram_detection(model, tokens, metric_fn)
print("Long-range heads:")
for l, h, dist in result['long_range_heads'][:5]:
    ratio = result['direct_vs_skip'][l, h]
    print(f"  L{l}H{h}: avg_distance={dist:.2f}, local_ratio={ratio:.2f}")

## Negative Mover Detection

In [ ]:
result = negative_mover_detection(model, tokens)
print("Most negative heads:")
for l, h, s in result['negative_heads'][:5]:
    print(f"  L{l}H{h}: min_logit={s:.4f}")
print("\nMost positive heads:")
for l, h, s in result['positive_heads'][:5]:
    print(f"  L{l}H{h}: max_logit={s:.4f}")

## Backup Circuit Detection

In [ ]:
result = backup_circuit_detection(model, tokens, metric_fn)
print(f"Redundancy score: {result['redundancy_score']:.4f}")
print(f"\nBackup pairs:")
for (l1, h1), (l2, h2), comp in result['backup_pairs'][:5]:
    print(f"  L{l1}H{h1} <-> L{l2}H{h2}: compensation={comp:.4f}")

## Signal Boosting

In [ ]:
result = signal_boosting_detection(model, tokens, metric_fn)
print(f"Boosting layers: {result['boosting_layers']}")
print(f"Attenuation layers: {result['attenuation_layers']}")
for i in range(cfg.n_layers):
    print(f"  Layer {i}: contribution={result['layer_contributions'][i]:+.4f}, "
          f"signal={result['signal_trajectory'][i]:.4f}")

## Motif Catalog

In [ ]:
result = motif_catalog(model, tokens, metric_fn)
print(f"Total motifs: {result['total_motifs']}")
print(f"Dominant: {result['dominant_motif']}")
for motif in result['motifs_found']:
    print(f"  {motif['type']}: {motif['description']}")